# Quickstart

alethia matches messy entity names against a reference list.

```bash
pip install git+https://github.com/saketlab/alethia.git#subdirectory=python
```


In [1]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

## Match against a reference list

The default matches on spelling and needs no model download.

In [2]:
from alethia import alethia

messy = ["New Yrok", "Los Angelos", "Chicagoo", "Houston"]
reference = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

alethia(messy, reference)[["given_entity", "alethia_prediction", "alethia_score"]]

,given_entity,alethia_prediction,alethia_score
0,New Yrok,New York,0.875000
1,Los Angelos,Los Angeles,0.909091
2,Chicagoo,Chicago,0.933333
3,Houston,Houston,1.000000


## Match on meaning

Pass `model=` a model name when the two vocabularies share little surface form.
The default cutoff of 0.7 is tuned for spelling, so cross-vocabulary matching
usually needs a lower one.

In [3]:
clinical = ["heart attack", "high blood sugar", "kidney stones"]
icd = ["Myocardial infarction", "Diabetes mellitus", "Calculus of kidney", "Asthma"]

alethia(clinical, icd, model="all-MiniLM-L6-v2", threshold=0.4)[
    ["given_entity", "alethia_prediction", "alethia_score"]
]

,given_entity,alethia_prediction,alethia_score
0,heart attack,Myocardial infarction,0.593563
1,high blood sugar,Diabetes mellitus,0.606745
2,kidney stones,Calculus of kidney,0.454118


## Choose a model without labels

`assess_models` scores candidates on your own data. The composite is relative,
so it needs at least two.

In [4]:
from alethia import assess_models

report = assess_models(
    queries=clinical,
    references=icd,
    models={"MiniLM": "all-MiniLM-L6-v2", "mpnet": "all-mpnet-base-v2"},
)
report.to_table()[["model", "score", "mean_margin_z", "mutual_nn_rate"]]

,model,score,mean_margin_z,mutual_nn_rate
0,mpnet,0.5,1.242463,1.0
1,MiniLM,-0.5,1.682301,1.0


In [5]:
report.best.name

'mpnet'

## Cluster without a reference list

Edges need mutual nearest-neighbour agreement, and each carries a confidence.

In [6]:
from alethia import cluster_entities

entities = ["Adidas AG", "adidas", "Nike Inc", "Nike, Inc.", "Puma SE"]
result = cluster_entities(entities, model="all-MiniLM-L6-v2")

for cid, members in result.clusters().items():
    print(f"{result.canonical[cid]:12} {members}")

adidas       ['Adidas AG', 'adidas']
Nike Inc     ['Nike Inc', 'Nike, Inc.']
Puma SE      ['Puma SE']


In [7]:
import pandas as pd

pd.DataFrame(result.edge_records())

,entity_a,entity_b,cosine,margin,mutual,confidence
0,Nike Inc,"Nike, Inc.",0.9731,0.3430,True,1.3068
1,Adidas AG,adidas,0.8955,0.2654,True,1.1331
